# NutriLeafNet: Vision Transformer for Cassava Leaf Nutrient-Deficiency Classification
Pure transformer pipeline (no CNN backbone). Run cell by cell in Google Colab (GPU runtime).

Expected dataset layout (one folder per class, real field photos):
  cassava_nutrient_dataset/
      healthy/            img001.jpg ...
      nitrogen_deficiency/ ...
      potassium_deficiency/ ...
      (your own class folders)


In [ ]:
# [1] Install + imports
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "scikit-learn", "seaborn", "matplotlib", "tqdm", "pillow"], check=False)

import os, json, random, time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms          # only used for image preprocessing/augmentation (not a CNN)
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
from transformers import ViTForImageClassification, get_cosine_schedule_with_warmup

In [ ]:
# [2] Configuration  (edit DATA_DIR before running)
@dataclass
class Config:
    data_dir: str = "/content/drive/MyDrive/cassava_nutrient_dataset"   # <-- your dataset folder
    out_dir: str = "/content/drive/MyDrive/nutrileafnet_outputs"        # models + plots are saved here
    model_name: str = "google/vit-base-patch16-224"
    img_size: int = 224
    batch_size: int = 32
    epochs: int = 20
    lr_backbone: float = 2e-5       # small LR: gently fine-tune pretrained transformer weights
    lr_head: float = 1e-3           # larger LR: new classification head learns from scratch
    weight_decay: float = 0.05
    label_smoothing: float = 0.1    # still cross-entropy; reduces over-confidence on noisy field images
    patience: int = 5               # early stopping patience (epochs)
    seed: int = 42
    num_workers: int = 2

cfg = Config()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Device:", device)

try:                                           # mount Google Drive when running in Colab
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
os.makedirs(cfg.out_dir, exist_ok=True)

In [ ]:
# [3] Module 1: dataset loading + stratified 70/15/15 split
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def scan_dataset(root):
    """Return (paths, labels, class_names). Class = sub-folder name."""
    root = Path(root)
    class_names = sorted(d.name for d in root.iterdir() if d.is_dir())
    paths, labels = [], []
    for idx, cname in enumerate(class_names):
        for p in sorted((root / cname).rglob("*")):
            if p.suffix.lower() in IMG_EXT:
                paths.append(str(p)); labels.append(idx)
    return paths, np.array(labels), class_names

paths, labels, class_names = scan_dataset(cfg.data_dir)
num_classes = len(class_names)
print(f"{len(paths)} images, {num_classes} classes: {class_names}")
for i, c in enumerate(class_names):
    print(f"  {c:30s} {(labels == i).sum()}")

# Stratified split: 70% train, then the remaining 30% split equally into 15% val / 15% test
tr_p, tmp_p, tr_y, tmp_y = train_test_split(paths, labels, test_size=0.30,
                                            stratify=labels, random_state=cfg.seed)
va_p, te_p, va_y, te_y = train_test_split(tmp_p, tmp_y, test_size=0.50,
                                          stratify=tmp_y, random_state=cfg.seed)
print(f"Train {len(tr_p)} | Val {len(va_p)} | Test {len(te_p)}")

In [ ]:
# [4] Module 1b: real-world augmentation (applied to TRAIN only)
class AddGaussianNoise:
    """Sensor-noise injection: adds N(0, sigma) noise to a [0,1] tensor, sigma random per image."""
    def __init__(self, sigma_range=(0.0, 0.06), p=0.5):
        self.sigma_range, self.p = sigma_range, p
    def __call__(self, t):
        if random.random() > self.p:
            return t
        sigma = random.uniform(*self.sigma_range)
        return (t + torch.randn_like(t) * sigma).clamp(0, 1)

MEAN, STD = [0.5, 0.5, 0.5], [0.5, 0.5, 0.5]    # normalisation used by google/vit-base-patch16-224

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(cfg.img_size, scale=(0.6, 1.0)),            # different distances / framing
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),                                           # camera angle variation
    transforms.ColorJitter(brightness=0.4, contrast=0.3, saturation=0.2, hue=0.02),  # lighting changes
    transforms.RandomApply([transforms.GaussianBlur(5, sigma=(0.1, 1.5))], p=0.2),    # slight focus blur
    transforms.ToTensor(),
    AddGaussianNoise(),                                                      # noise injection
    transforms.Normalize(MEAN, STD),
])
eval_tfms = transforms.Compose([
    transforms.Resize((cfg.img_size, cfg.img_size)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class LeafDataset(Dataset):
    def __init__(self, paths, labels, tfm):
        self.paths, self.labels, self.tfm = list(paths), list(labels), tfm
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.tfm(img), int(self.labels[i])

def make_loader(p, y, tfm, shuffle):
    return DataLoader(LeafDataset(p, y, tfm), batch_size=cfg.batch_size, shuffle=shuffle,
                      num_workers=cfg.num_workers, pin_memory=use_amp)

train_loader = make_loader(tr_p, tr_y, train_tfms, True)
val_loader = make_loader(va_p, va_y, eval_tfms, False)
test_loader = make_loader(te_p, te_y, eval_tfms, False)

# Preview augmentations
def show_augmentations(n=6):
    fig, axes = plt.subplots(2, n // 2, figsize=(12, 5))
    base = Image.open(tr_p[0]).convert("RGB")
    for ax in axes.ravel():
        x = train_tfms(base) * 0.5 + 0.5
        ax.imshow(x.permute(1, 2, 0).clamp(0, 1)); ax.axis("off")
    plt.suptitle("Augmented versions of one field photo"); plt.show()
show_augmentations()

In [ ]:
# [5] Module 2: Vision Transformer model (no CNN backbone)
# The image is cut into 16x16 patches -> linearly embedded -> processed by 12 self-attention layers.
# "eager" attention is required so we can read attention weights for explainability.
model = ViTForImageClassification.from_pretrained(
    cfg.model_name,
    num_labels=num_classes,
    id2label={i: c for i, c in enumerate(class_names)},
    label2id={c: i for i, c in enumerate(class_names)},
    ignore_mismatched_sizes=True,          # replace the 1000-class ImageNet head with our own head
    attn_implementation="eager",
).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f} M")

In [ ]:
# [6] Module 3: training strategy (AdamW + cross-entropy + EarlyStopping + ModelCheckpoint)
# Class weights counter class imbalance
counts = np.bincount(tr_y, minlength=num_classes)
class_w = torch.tensor(len(tr_y) / (num_classes * np.maximum(counts, 1)), dtype=torch.float, device=device)
criterion = nn.CrossEntropyLoss(weight=class_w, label_smoothing=cfg.label_smoothing)

optimizer = torch.optim.AdamW([
    {"params": model.vit.parameters(), "lr": cfg.lr_backbone},
    {"params": model.classifier.parameters(), "lr": cfg.lr_head},
], weight_decay=cfg.weight_decay)

total_steps = cfg.epochs * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

BEST_DIR = os.path.join(cfg.out_dir, "best_model")

class EarlyStopping:
    """Stop training when the monitored metric has not improved for `patience` epochs."""
    def __init__(self, patience): self.patience, self.best, self.bad = patience, -1e9, 0
    def step(self, score):
        if score > self.best: self.best, self.bad = score, 0
        else: self.bad += 1
        return self.bad >= self.patience

class ModelCheckpoint:
    """Save the model whenever the monitored metric (val macro-F1) improves."""
    def __init__(self, path): self.path, self.best = path, -1e9
    def step(self, score, model):
        if score > self.best:
            self.best = score
            model.save_pretrained(self.path)        # config (incl. class names) + weights
            return True
        return False

def run_epoch(loader, train):
    model.train(train)
    total, preds, gts = 0.0, [], []
    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        with torch.set_grad_enabled(train):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(pixel_values=x).logits
                loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
        total += loss.item() * x.size(0)
        preds += logits.argmax(1).tolist(); gts += y.tolist()
    f1 = precision_recall_fscore_support(gts, preds, average="macro", zero_division=0)[2]
    return total / len(loader.dataset), accuracy_score(gts, preds), f1

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}
early, ckpt = EarlyStopping(cfg.patience), ModelCheckpoint(BEST_DIR)

for epoch in range(1, cfg.epochs + 1):
    t0 = time.time()
    tl, ta, _ = run_epoch(train_loader, True)
    vl, va, vf = run_epoch(val_loader, False)
    for k, v in zip(history, [tl, vl, ta, va, vf]): history[k].append(v)
    saved = ckpt.step(vf, model)
    print(f"Epoch {epoch:02d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f} "
          f"F1 {vf:.3f} | {time.time()-t0:.0f}s {'[saved best]' if saved else ''}")
    if early.step(vf):
        print("Early stopping triggered."); break

# Training curves
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="val")
ax[1].set_title("Accuracy"); ax[1].legend()
plt.savefig(os.path.join(cfg.out_dir, "training_curves.png"), dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# [7] Module 4: evaluation on the held-out TEST set (never seen during training/validation)
model = ViTForImageClassification.from_pretrained(BEST_DIR, attn_implementation="eager").to(device).eval()

@torch.no_grad()
def predict_all(loader):
    preds, gts = [], []
    for x, y in tqdm(loader, leave=False):
        preds += model(pixel_values=x.to(device)).logits.argmax(1).cpu().tolist(); gts += y.tolist()
    return np.array(gts), np.array(preds)

y_true, y_pred = predict_all(test_loader)
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
print(f"Test Accuracy : {acc:.4f}\nMacro Precision: {p:.4f}\nMacro Recall   : {r:.4f}\nMacro F1       : {f1:.4f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(1.2 * num_classes + 3, 1.0 * num_classes + 3))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix (test set)")
plt.savefig(os.path.join(cfg.out_dir, "confusion_matrix.png"), dpi=150, bbox_inches="tight"); plt.show()

json.dump({"accuracy": acc, "macro_precision": p, "macro_recall": r, "macro_f1": f1},
          open(os.path.join(cfg.out_dir, "test_metrics.json"), "w"), indent=2)

In [ ]:
# [8] Module 5: explainability - attention rollout (replaces Grad-CAM, which needs conv feature maps)
@torch.no_grad()
def attention_rollout(model, pixel_values):
    """
    Attention rollout (Abnar & Zuidema, 2020).
    1) average attention over heads in every layer
    2) add identity (residual connection) and re-normalise rows
    3) multiply layer matrices together -> how much each patch influences the [CLS] token
    Returns a (grid x grid) map per image, scaled to [0, 1].
    """
    out = model(pixel_values=pixel_values, output_attentions=True)
    attn = torch.stack(out.attentions).mean(dim=2)               # (layers, B, tokens, tokens)
    attn = attn + torch.eye(attn.size(-1), device=attn.device)
    attn = attn / attn.sum(dim=-1, keepdim=True)
    rollout = attn[0]
    for layer in attn[1:]:
        rollout = layer @ rollout
    mask = rollout[:, 0, 1:]                                     # [CLS] -> patch tokens
    g = int(mask.size(-1) ** 0.5)
    mask = mask.reshape(-1, g, g)
    mask = (mask - mask.amin((1, 2), keepdim=True)) / (mask.amax((1, 2), keepdim=True) - mask.amin((1, 2), keepdim=True) + 1e-8)
    return out.logits, mask

def overlay(img_pil, mask):
    m = F.interpolate(mask[None, None].float().cpu(), size=(cfg.img_size, cfg.img_size),
                      mode="bicubic", align_corners=False)[0, 0].clamp(0, 1).numpy()
    base = np.asarray(img_pil.resize((cfg.img_size, cfg.img_size))) / 255.0
    heat = plt.cm.jet(m)[..., :3]
    return np.clip(0.55 * base + 0.45 * heat, 0, 1)

def visualize_attention(n=8):
    idx = random.sample(range(len(te_p)), min(n, len(te_p)))
    fig, axes = plt.subplots(2, len(idx), figsize=(3 * len(idx), 6.5))
    for col, i in enumerate(idx):
        img = Image.open(te_p[i]).convert("RGB")
        x = eval_tfms(img).unsqueeze(0).to(device)
        logits, mask = attention_rollout(model, x)
        pred = logits.argmax(1).item()
        axes[0, col].imshow(img.resize((cfg.img_size, cfg.img_size))); axes[0, col].axis("off")
        axes[0, col].set_title(f"True: {class_names[te_y[i]]}", fontsize=8)
        axes[1, col].imshow(overlay(img, mask[0])); axes[1, col].axis("off")
        axes[1, col].set_title(f"Pred: {class_names[pred]}", fontsize=8,
                               color="green" if pred == te_y[i] else "red")
    plt.suptitle("ViT attention rollout (red = regions the model focused on)")
    plt.savefig(os.path.join(cfg.out_dir, "attention_maps.png"), dpi=150, bbox_inches="tight"); plt.show()

visualize_attention()

In [ ]:
# [9] Save artefacts for the Streamlit app
# best_model/ (weights + config with class names) is already saved in cfg.out_dir.
print("Download this folder for the app:", BEST_DIR)
print("Files:", os.listdir(BEST_DIR))